# Stichprobe: check the claims by eye

`evaluation.ipynb`, `real_photos.ipynb` and `PASSPORT_VALIDATOR.md` report how the checks do over thousands of photos. This notebook draws a random sample of those photos, runs the validator on each one as the code stands now, and shows every photo next to what it should be and what the validator said, so a person can check the claims without trusting the summaries.

**The sample.** From the London Set evaluation (`scripts/build_eval_set.py`) only the **test half** (even ids), because the thresholds were set on the other half: `N_SHOULD_PASS` compliant crops, plus `N_PER_VARIANT` photos of each variant. A defect variant breaks exactly one rule; the must-pass variants (`roll_4`, `clear_glasses`) should be valid like the compliant crops. From the real photos (`scripts/build_openimages_set.py`: candid Flickr photos, not ID photos) `N_REAL` judgeable ones; they have no single expected verdict, so judge them by eye against the labels shown with them.

Change `SEED` (or set `STICHPROBE_SEED`) for a different draw; the sizes are at the top of the next cell. The photo sets are not in the repository: `LONDON_SET` and `OPENIMAGES_SET` point at them, by default `../datasets/frl-london-eval` and `../datasets/openimages-faces/eval` (see the README for how to build them). The notebook also works on a set made by `scripts/make_test_photos.py`. It shows photos, so clear its outputs before committing it.

**How to read a tile.** Bold: the photo. Then what it should be, then what the validator said: *valid*, *warns* or *fails*, with the rules concerned and the validator's reasons. For a defect variant the verdict line is about the rule the variant breaks; other rules the photo fails are listed as *also*. A green ✓ marks a tile where the verdict agrees with the label, an amber ~ one where the validator only warns, a red ✗ one where it disagrees: those are the ones to look at.

In [ ]:
import base64
import contextlib
import io
import os
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from IPython.display import HTML, display
from PIL import Image

from passphoto_validation.core import PassportPhotoValidator
from passphoto_validation.core.rules import RULES

SEED = int(os.environ.get('STICHPROBE_SEED', 2026))
N_SHOULD_PASS = 12  # compliant crops
N_PER_VARIANT = 2   # photos per variant (defect and must-pass)
N_REAL = 16         # real photos

LONDON = Path(os.environ.get('LONDON_SET', Path.cwd().parents[1] / 'datasets' / 'frl-london-eval'))
OPENIMAGES = Path(os.environ.get('OPENIMAGES_SET', Path.cwd().parents[1] / 'datasets' / 'openimages-faces' / 'eval'))
# OFIQ cloned next to this repo, unless OFIQ_CONFIG_DIR says otherwise
OFIQ_DATA = Path.cwd().parents[1] / 'OFIQ-Project' / 'data'
if 'OFIQ_CONFIG_DIR' not in os.environ and OFIQ_DATA.is_dir():
    os.environ['OFIQ_CONFIG_DIR'] = str(OFIQ_DATA)

TITLES = {rule.id: rule.title for rule in RULES}
rng = np.random.default_rng(SEED)


@contextlib.contextmanager
def quiet_native_logs():
    """Hide MediaPipe's C++ log lines, which go straight to the process's stderr."""
    saved = os.dup(2)
    with open(os.devnull, 'w') as devnull:
        os.dup2(devnull.fileno(), 2)
        try:
            yield
        finally:
            os.dup2(saved, 2)
            os.close(saved)


with quiet_native_logs():
    validator = PassportPhotoValidator()
print('OFIQ:', 'loaded' if validator.ofiq else validator.ofiq_unavailable)
print(f'seed {SEED}; London Set at {LONDON}; real photos at {OPENIMAGES}')

## 1. The draw

In [ ]:
def draw(frame, n):
    return frame.sample(n=min(n, len(frame)), random_state=int(rng.integers(1 << 31)))


manifest = pd.read_csv(LONDON / 'manifest.csv', keep_default_na=False)
test = manifest[manifest['split'] == 'test'] if 'split' in manifest and (manifest['split'] == 'test').any() else manifest
should_pass = draw(test[test['variant'] == 'compliant'], N_SHOULD_PASS)
variants = sorted(v for v in test['variant'].unique() if v != 'compliant')
per_variant = pd.concat([draw(test[test['variant'] == v], N_PER_VARIANT) for v in variants])

if OPENIMAGES.is_dir():
    labels = pd.read_json(OPENIMAGES / 'vlm_labels.jsonl', lines=True).set_index('file')
    credits = pd.read_csv(OPENIMAGES / 'credits.csv').set_index('file')
    real = draw(labels[labels['face_judgeable'] != False], N_REAL)
else:
    labels = credits = real = pd.DataFrame()
    print(f'no real photos: {OPENIMAGES} does not exist')

print(f'{len(should_pass)} compliant crops, {len(per_variant)} photos of {len(variants)} variants, {len(real)} real photos')

## 2. Run the validator

Every sampled photo goes through the whole pipeline as it is now, so this reflects the code, not an older results file. About two seconds per photo.

In [ ]:
def verdict(path):
    with quiet_native_logs():
        report = validator(cv2.imread(str(path)), crop=False)
    return dict(
        is_valid=report.is_valid,
        failed=[r.id for r in report.rules if r.status == 'fail'],
        warned=[r.id for r in report.rules if r.status == 'warn'],
        status={r.id: r.status for r in report.rules},
        reasons={r.id: r.reasons for r in report.rules if r.status != 'pass'},
    )


rows = []
for _, row in pd.concat([should_pass, per_variant]).iterrows():
    path = LONDON / row['file']
    rows.append(dict(set='london', file=row['file'], variant=row['variant'], expected=row['broken_rule'] or 'valid',
                     path=path, **verdict(path)))
for file in real.index:
    path = OPENIMAGES / file
    rows.append(dict(set='real', file=file, variant='real', expected='', path=path, **verdict(path)))
sample = pd.DataFrame(rows)


def agreement(row):
    """'yes', 'warn' (flagged but not failed) or 'no'; '' for real photos, which have no single expected verdict."""
    if row['expected'] == '':
        return ''
    if row['expected'] == 'valid':
        return 'no' if row['failed'] else 'yes'
    status = row['status'].get(row['expected'], 'skipped')
    return {'fail': 'yes', 'warn': 'warn'}.get(status, 'no')


sample['agreement'] = sample.apply(agreement, axis=1)
counts = sample[sample['set'] == 'london']['agreement'].value_counts()
print(f"London photos: {counts.get('yes', 0)} agree, {counts.get('warn', 0)} warn only, {counts.get('no', 0)} disagree")

In [ ]:
MARK = {'yes': ('✓', '#2a7a3b'), 'warn': ('~', '#a86b12'), 'no': ('✗', '#b3261e'), '': ('', '#6b6a66')}
MUTED = '#6b6a66'


def thumbnail(path, width=210):
    image = Image.open(path).convert('RGB')
    image.thumbnail((width, int(width * 9 / 7)))
    buffer = io.BytesIO()
    image.save(buffer, 'JPEG', quality=82)
    return 'data:image/jpeg;base64,' + base64.b64encode(buffer.getvalue()).decode()


def shorten(texts, limit=170):
    text = ' '.join(texts)
    return text if len(text) <= limit else text[:limit - 1] + '…'


def verdict_lines(row):
    """(expectation, verdict, also, reasons) for one sampled photo."""
    failed, warned, status, reasons = row['failed'], row['warned'], row['status'], row['reasons']
    if row['expected'] == '':
        expectation = ''
        if failed:
            verdict = f"fails {', '.join(failed)}"
        elif warned:
            verdict = f"valid, warns {', '.join(warned)}"
        else:
            verdict = 'valid'
        return expectation, verdict, '', shorten(sum((reasons[r] for r in failed + warned), []))
    if row['expected'] == 'valid':
        expectation = 'should be valid'
        if failed:
            verdict = f"fails {', '.join(failed)}"
        elif warned:
            verdict = f"valid, warns {', '.join(warned)}"
        else:
            verdict = 'valid'
        return expectation, verdict, '', shorten(sum((reasons[r] for r in failed + warned), []))
    rule = row['expected']
    expectation = f'should fail {rule}'
    verdict = {'fail': f'fails {rule}', 'warn': f'warns {rule}', 'pass': f'passes {rule}'}.get(status.get(rule), f'{rule} skipped')
    also = ', '.join(r for r in failed if r != rule)
    return expectation, verdict, f'also fails {also}' if also else '', shorten(reasons.get(rule, []))


def tile(row, extra=''):
    mark, colour = MARK[row['agreement']]
    expectation, verdict, also, reasons = verdict_lines(row)
    title = row['file'].replace('.jpg', '')
    lines = [f'<div style="font-weight:600">{title}</div>']
    if extra:
        lines.append(f'<div style="color:{MUTED}">{extra}</div>')
    if expectation:
        lines.append(f'<div>{expectation}</div>')
    lines.append(f'<div style="color:{colour};font-weight:600">{mark} {verdict}</div>')
    if also:
        lines.append(f'<div style="color:{MUTED}">{also}</div>')
    if reasons:
        lines.append(f'<div style="color:{MUTED};font-size:11px">{reasons}</div>')
    return (f'<div style="width:210px"><img src="{thumbnail(row["path"])}" style="width:210px;display:block;border-radius:4px">'
            f'<div style="font:12px/1.35 system-ui,sans-serif;margin-top:4px;word-break:break-word">{"".join(lines)}</div></div>')


def grid(rows, heading=None, extras=None):
    tiles = ''.join(tile(row, extras(row) if extras else '') for _, row in rows.iterrows())
    head = f'<h4 style="font:600 14px system-ui,sans-serif;margin:18px 0 8px">{heading}</h4>' if heading else ''
    display(HTML(f'{head}<div style="display:flex;flex-wrap:wrap;gap:14px">{tiles}</div>'))

## 3. Photos that should be valid

Compliant crops of the neutral front photos, then the must-pass variants: a head rolled 4° (within tolerance) and clear glasses. Every one of them should come out *valid*; a warning is allowed, a fail is a disagreement.

In [ ]:
london = sample[sample['set'] == 'london']
grid(london[london['variant'] == 'compliant'], 'Compliant crops')
for variant in [v for v in variants if (test[test['variant'] == v]['broken_rule'] == '').all()]:
    grid(london[london['variant'] == variant], f'{variant} (must pass)')

## 4. Photos with one known defect

Each variant breaks exactly one rule, named in the heading with the rule's title. The verdict line is about that rule. The smiling and 3/4-view variants are real photos of the same people; the rest are drawn into the compliant crop.

In [ ]:
for variant in [v for v in variants if (test[test['variant'] == v]['broken_rule'] != '').any()]:
    rule = test[test['variant'] == variant]['broken_rule'].iloc[0]
    grid(london[london['variant'] == variant], f'{variant} → {rule}: {TITLES.get(rule, rule)}')

## 5. Real photos

Candid photos with one large face. The line under the name lists what the labels say a clerk would object to (nothing listed: the labels found nothing wrong with the face; the background, framing and lighting are not labelled). Most candid photos rightly fail several rules, so the question here is whether each fail is deserved and whether anything the labels list was missed.

In [ ]:
def label_summary(file):
    row = labels.loc[file]
    notes = []
    if row['glasses'] != 'none':
        notes.append(f"{row['glasses']} glasses")
    if row['glasses_glare']:
        notes.append('reflections')
    if row['frame_covers_eyes']:
        notes.append('frame over the eyes')
    if row['hair_over_eyes']:
        notes.append('hair over the eyes')
    if row['eyes_open'] == False:
        notes.append('eyes not open')
    if row['gaze_at_camera'] == False:
        notes.append('looks away')
    if row['mouth_closed'] == False:
        notes.append('mouth open')
    if row['expression_neutral'] == False:
        notes.append('not neutral')
    if row['head_covering'] != 'none':
        notes.append(str(row['head_covering']))
    if row['head_tilted']:
        notes.append('tilted')
    if row['head_turned']:
        notes.append('turned')
    if row['red_eyes']:
        notes.append('red eyes')
    author = credits.loc[file, 'author'] if file in credits.index else ''
    return f"labels: {', '.join(notes) or 'nothing'} · photo: {author}"


if len(real):
    grid(sample[sample['set'] == 'real'], 'Real photos', extras=lambda row: label_summary(row['file']))

## 6. The sample against the whole test half

The same counts for the whole test half, from `results.jsonl` (written by `scripts/run_eval.py`; re-run it after changing a check), so a surprising tile can be put in proportion. *flagged* means the rule fails or warns.

In [ ]:
if not (LONDON / 'results.jsonl').is_file():
    raise SystemExit(f'no results.jsonl in {LONDON}: run scripts/run_eval.py --set {LONDON} first to compare the sample with the whole set')
full = pd.read_json(LONDON / 'results.jsonl', lines=True)
full = full[full['split'] == 'test'] if (full['split'] == 'test').any() else full
full['failed'] = full['rules'].apply(lambda rules: [r for r, s in rules.items() if s == 'fail'])
full['warned'] = full['rules'].apply(lambda rules: [r for r, s in rules.items() if s == 'warn'])

rows = []
for variant in ['compliant'] + variants:
    rule = test[test['variant'] == variant]['broken_rule'].iloc[0]
    s, f = london[london['variant'] == variant], full[full['variant'] == variant]
    if rule == '':
        rows.append(dict(variant=variant, expected='valid', sample_n=len(s),
                         sample_agree=(s['failed'].str.len() == 0).sum(), sample_clean=((s['failed'].str.len() == 0) & (s['warned'].str.len() == 0)).sum(),
                         full_n=len(f), full_agree=(f['failed'].str.len() == 0).mean(),
                         full_clean=((f['failed'].str.len() == 0) & (f['warned'].str.len() == 0)).mean()))
    else:
        rows.append(dict(variant=variant, expected=f'fails {rule}', sample_n=len(s),
                         sample_agree=s['failed'].apply(lambda failed: rule in failed).sum(),
                         sample_flagged=s.apply(lambda row: rule in row['failed'] or rule in row['warned'], axis=1).sum(),
                         full_n=len(f), full_agree=f['rules'].apply(lambda rules: rules.get(rule) == 'fail').mean(),
                         full_flagged=f['rules'].apply(lambda rules: rules.get(rule) in ('fail', 'warn')).mean()))
summary = pd.DataFrame(rows).set_index('variant')
summary = summary[['expected', 'sample_n', 'sample_agree', 'sample_clean', 'sample_flagged', 'full_n', 'full_agree', 'full_clean', 'full_flagged']]
summary.style.format({c: '{:.0%}' for c in ('full_agree', 'full_clean', 'full_flagged')}, na_rep='').format(
    {c: '{:.0f}' for c in ('sample_clean', 'sample_flagged')}, na_rep='')

## 7. Check sheet

One row per sampled photo, for notes. To report a disagreement, the file name is enough: `scripts/build_eval_set.py` makes the same photo again from the London Set, and `uv run passport-validator validate <photo> -v` shows every check's measurements for it.

In [ ]:
sheet = sample.copy()
sheet['verdict'] = sheet.apply(lambda row: verdict_lines(row)[1], axis=1)
sheet['also'] = sheet.apply(lambda row: verdict_lines(row)[2], axis=1)
sheet['reasons'] = sheet.apply(lambda row: verdict_lines(row)[3], axis=1)
sheet['agreement'] = sheet['agreement'].map({'yes': '✓', 'warn': '~', 'no': '✗', '': ''})
with pd.option_context('display.max_rows', 200, 'display.max_colwidth', 160):
    display(sheet[['file', 'variant', 'expected', 'agreement', 'verdict', 'also', 'reasons']].set_index('file'))